# Method A (Direct Signal Classifier) - Optimization & Production

**Current Performance**: +23.43% return (2 trades) - Best performer but needs investigation

**Goal**: Transform research model into production-ready trading system

---

## 📋 Available Optimization Paths

**Research & Development:**
1. [Signal Generation Analysis](#1-signal-generation-analysis) - Why so few trades?
2. [Feature Engineering](#2-feature-engineering) - Add more indicators
3. [Model Architecture](#3-model-architecture) - Try different networks
4. [Multi-Timeframe Strategy](#4-multi-timeframe-strategy) - Combine daily + weekly
5. [Walk-Forward Validation](#5-walk-forward-validation) - Proper out-of-sample testing
6. [Ensemble Methods](#6-ensemble-methods) - Combine multiple models

**Production Ready:**
7. [Trader-Friendly Interface](#7-trader-friendly-interface) - Easy to use for non-technical users
8. [Live Trading Preparation](#8-live-trading-preparation) - Paper trading & real-time
9. [Performance Monitoring](#9-performance-monitoring) - Automated alerts & dashboards
10. [Configuration Management](#10-configuration-management) - Centralized parameter control

---

**Instructions**: Uncomment and run sections as needed. Each section is independent.

## Setup & Imports

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, GRU, Attention
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Technical analysis
import talib

# Custom modules
from Data_Handling import get_data_auto

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 8)

print("✓ Libraries imported")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Libraries imported
📅 Date: 2026-01-20 20:29:41


## Model Configuration & Architecture

Define the Method A model (3-class signal classifier: BUY/HOLD/SELL)

In [ ]:
# Model hyperparameters (easy to experiment with)
LOOKBACK_WINDOW = 60  # Number of days to look back
LSTM_UNITS_1 = 64     # First LSTM layer units
LSTM_UNITS_2 = 64     # Second LSTM layer units
DROPOUT_RATE = 0.3    # Dropout rate for regularization
LEARNING_RATE = 0.001 # Adam optimizer learning rate
BATCH_SIZE = 32       # Training batch size
EPOCHS = 50           # Maximum training epochs
PATIENCE = 10         # Early stopping patience

print("Model Hyperparameters:")
print(f"  Lookback Window: {LOOKBACK_WINDOW}")
print(f"  LSTM Units: [{LSTM_UNITS_1}, {LSTM_UNITS_2}]")
print(f"  Dropout Rate: {DROPOUT_RATE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Max Epochs: {EPOCHS}")

In [ ]:
def build_method_a_model(input_shape, units_1=64, units_2=64, dropout=0.3, lr=0.001):
    """
    Build Method A: 3-class signal classifier
    
    Args:
        input_shape: Tuple (lookback_window, num_features)
        units_1: First LSTM layer units
        units_2: Second LSTM layer units
        dropout: Dropout rate
        lr: Learning rate
    
    Returns:
        Compiled Keras model
    """
    model = Sequential([
        LSTM(units_1, return_sequences=True, input_shape=input_shape, name='lstm_1'),
        Dropout(dropout, name='dropout_1'),
        LSTM(units_2, name='lstm_2'),
        Dropout(dropout, name='dropout_2'),
        Dense(3, activation='softmax', name='output')  # BUY=0, HOLD=1, SELL=2
    ], name='Method_A_Classifier')
    
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

print("✓ Model builder function defined")

In [ ]:
# Training callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    'models/method_a_best.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

print("✓ Training callbacks configured")